<a href="https://colab.research.google.com/github/Csahoo009/ML-for-DA/blob/main/T5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files

# Load dataset directly from uploaded CSV
uploaded = files.upload()

import pandas as pd
df = pd.read_csv("indian_house_prices.csv")

# Scale Price to ₹ Lakhs (1 Lakh = 100,000 INR)
df["Price_Lakhs"] = df["Price"] / 100000.0

print(f"Total records: {df.shape[0]}, Total features: {df.shape[1]}")
print(df[["City", "Location", "Area", "No. of Bedrooms", "Resale", "Price_Lakhs"]].head())

Saving indian_house_prices.csv to indian_house_prices.csv
Total records: 32963, Total features: 42
        City                         Location  Area  No. of Bedrooms  Resale  \
0  Bangalore                 JP Nagar Phase 1  3340                4       0   
1  Bangalore       Dasarahalli on Tumkur Road  1045                2       0   
2  Bangalore  Kannur on Thanisandra Main Road  1179                2       0   
3  Bangalore                     Doddanekundi  1675                3       0   
4  Bangalore                          Kengeri  1670                3       0   

   Price_Lakhs  
0       300.00  
1        78.88  
2        48.66  
3        83.58  
4        68.45  


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

# 1. Feature Subsets Configuration
# Model 1: Core Property Attributes (3 Features)
core_features = ['Area', 'No. of Bedrooms', 'Resale']

# Model 2: Core Attributes + Key Amenities (5 Features)
extended_features = ['Area', 'No. of Bedrooms', 'Resale', 'SwimmingPool', 'Gymnasium']

# Model 3: Comprehensive Multi-Variable Feature Set (10 Features)
full_features = [
    'Area', 'No. of Bedrooms', 'Resale', 'SwimmingPool', 'Gymnasium',
    'LiftAvailable', '24X7Security', 'PowerBackup', 'CarParking', 'ClubHouse'
]

target_col = 'Price_Lakhs'

In [ ]:
# 2. Performance Matrix Computation Function
def train_and_evaluate_mlr(feature_list, name):
    X = df[feature_list].fillna(df[feature_list].mean())
    y = df[target_col].fillna(df[target_col].mean())

    # Train-Test Split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Model Fitting
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Error & Accuracy Metrics
    n = len(y_test)
    k = len(feature_list)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    r2 = r2_score(y_test, y_pred)
    adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - k - 1))

    # 5-Fold Cross-Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2 = cross_val_score(model, X, y, cv=kf, scoring='r2').mean()

    return {
        "Num Features": k,
        "MAE (₹ Lakhs)": round(mae, 2),
        "MSE": round(mse, 2),
        "RMSE (₹ Lakhs)": round(rmse, 2),
        "MAPE (%)": f"{round(mape, 2)}%",
        "R² Score": round(r2, 4),
        "Adjusted R²": round(adj_r2, 4),
        "5-Fold CV R²": round(cv_r2, 4)
    }

In [ ]:
# 3. Generate the Evaluation Comparison Table
results = [
    train_and_evaluate_mlr(core_features, "Core (3 Features)"),
    train_and_evaluate_mlr(extended_features, "Extended (5 Features)"),
    train_and_evaluate_mlr(full_features, "Full (10 Features)")
]

t5_performance_matrix = pd.DataFrame(results, index=[
    "MLR: Core (Area, BHK, Resale)",
    "MLR: Extended (+ Pool, Gym)",
    "MLR: Full (+ Security, Parking, Lift, etc.)"
])

print("=== T5: MULTIPLE LINEAR REGRESSION PERFORMANCE MATRIX ===")
print(t5_performance_matrix.to_markdown())

=== T5: MULTIPLE LINEAR REGRESSION PERFORMANCE MATRIX ===
|                                             |   Num Features |   MAE (₹ Lakhs) |     MSE |   RMSE (₹ Lakhs) | MAPE (%)   |   R² Score |   Adjusted R² |   5-Fold CV R² |
|:--------------------------------------------|---------------:|----------------:|--------:|-----------------:|:-----------|-----------:|--------------:|---------------:|
| MLR: Core (Area, BHK, Resale)               |              3 |           80.36 | 48315.2 |           219.81 | 101.04%    |     0.1161 |        0.1157 |         0.0946 |
| MLR: Extended (+ Pool, Gym)                 |              5 |           79.93 | 48172.9 |           219.48 | 102.02%    |     0.1187 |        0.118  |         0.0969 |
| MLR: Full (+ Security, Parking, Lift, etc.) |             10 |           79.93 | 48158.3 |           219.45 | 102.02%    |     0.1189 |        0.1176 |         0.0971 |
